# Tech Challenge - Fase 3 | Previsao de evasao de estudantes

**FIAP - Pos-Graduacao em Machine Learning Engineering**

Este notebook **reproduz as analises principais** do projeto, na ordem em que elas
aconteceram:

1. Base processada (Fase 1)
2. Analise exploratoria
3. A correcao das notas
4. Cenarios de features
5. Comparacao de modelos (Fase 2)
6. Modelo final e avaliacao no teste (Fase 3)
7. Diagnostico de overfitting / underfitting
8. Demonstracao de inferencia

> Os artefatos pesados (comparacao de modelos, curvas) sao **gerados pelos scripts**
> `src/train.py` e `src/select_model.py` e aqui sao **lidos e interpretados**. Isso mantem o
> notebook rapido e garante que o numero mostrado seja o mesmo do relatorio final.


## 0. Configuracao do ambiente

In [1]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve
from sklearn.model_selection import StratifiedKFold

from src import config
from src.data_loader import load_processed_data
from src.evaluate import (
    compute_threshold_metrics,
    make_scorers,
    plot_precision_recall_curve,
    plot_roc_curve,
)
from src.train import create_or_load_split, split_train_holdout

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

print("RANDOM_STATE :", config.RANDOM_STATE)
print("Teste        :", f"{config.TEST_SIZE:.0%} (estratificado)")
print("Folds        :", config.N_SPLITS)
print("Metrica alvo :", config.PRIMARY_SCORING.upper())

RANDOM_STATE : 42
Teste        : 30% (estratificado)
Folds        : 5
Metrica alvo : F2


## 1. Base processada

A base bruta tem 4.424 linhas e 28 colunas. A Fase 1 removeu 1 duplicata exata e criou as
colunas de alvo, resultando em **4.423 linhas x 31 colunas**.

In [2]:
df = load_processed_data()
print("Dimensao:", df.shape)
print("Valores ausentes:", int(df.isna().sum().sum()), "(apenas nas colunas de alvo derivadas)")
df[["target_class", "target", "target_excl_enrolled"]].value_counts(dropna=False)

Dimensao: (4423, 31)
Valores ausentes: 794 (apenas nas colunas de alvo derivadas)


target_class  target  target_excl_enrolled
Graduado      0       0.0                     2209
Desistente    1       1.0                     1420
Matriculado   0       NaN                      794
Name: count, dtype: int64

## 2. Analise exploratoria - variavel-alvo

O `Target` original tem **3 classes**, mas o enunciado pede classificacao binaria (D1):
`Desistente = 1` contra `Graduado + Matriculado = 0`.

In [3]:
contagem = df["target_class"].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
contagem.plot(kind="bar", ax=axes[0], color="#4C72B0", rot=0, title="3 classes originais")
df["target"].value_counts().sort_index().plot(
    kind="bar", ax=axes[1], color="#DD8452", rot=0,
    title="Alvo binario (D1): 1 = evasao",
)
for ax in axes:
    ax.set_ylabel("Alunos")
axes[0].set_xlabel("")
axes[1].set_xlabel("")
plt.tight_layout()
plt.show()

taxa = df["target"].mean()
print(f"Evasoes: {int(df['target'].sum())} de {len(df)} ({taxa:.1%})")

Evasoes: 1420 de 4423 (32.1%)


/tmp/ipykernel_200314/2075742310.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. A correcao das notas (o achado mais importante da base)

As colunas `...SemestreGrau` vinham com valores impossiveis (ate `1.7e16`), **consistentes
com a perda do separador decimal**. A regra aplicada foi dividir por 10 enquanto o valor
exceder 20.

In [4]:
relatorio = json.loads(config.QUALITY_REPORT_JSON.read_text(encoding="utf-8"))

linhas = []
for coluna, stats in relatorio["grade_correction"]["columns"].items():
    linhas.append({
        "coluna": coluna.split("Semestre")[1],
        "valores_corrigidos": stats["n_corrected"],
        "pct": f"{stats['pct_corrected']:.1f}%",
        "max_antes": f"{stats['max_before']:.3g}",
        "max_depois": f"{stats['max_after']:.3f}",
    })
display(pd.DataFrame(linhas))

print("Exemplos 'antes -> depois':")
for exemplo in relatorio["grade_correction"]["columns"][config.GRADE_COLUMNS[0]]["examples"][:5]:
    print(f"   {exemplo['before']:.6g}  ->  {exemplo['after']:.4f}")

,coluna,valores_corrigidos,pct,max_antes,max_depois
0,Grau,1790,40.5%,1.73e+16,18.875
1,Grau,1675,37.9%,1.86e+16,18.571


Exemplos 'antes -> depois':
   1.34286e+16  ->  13.4286
   1.23333e+16  ->  12.3333
   1.18571e+16  ->  11.8571
   13875  ->  13.8750
   1.23333e+16  ->  12.3333


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, coluna in zip(axes, config.GRADE_COLUMNS):
    df[coluna].plot(kind="hist", bins=40, ax=ax, color="#55A868")
    ax.set_title(f"{coluna.split('Semestre')[1]} (corrigida)")
    ax.set_xlabel("Nota (0-20)")
plt.tight_layout()
plt.show()

/tmp/ipykernel_200314/1010357883.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Cenarios de features (D2)

- **Early Warning**: cadastro, socioeconomico, financeiro, ingresso e 1o semestre.
  E o candidato ao **deploy**, pois permite intervir mais cedo.
- **Completo**: acrescenta o 2o semestre. Serve de **benchmark**.

In [6]:
resumo = pd.DataFrame([
    {
        "cenario": dados["label"],
        "features": len(dados["columns"]),
        "descricao": dados["description"],
    }
    for dados in config.SCENARIOS.values()
])
display(resumo)

,cenario,features,descricao
0,Early Warning,24,"Cadastro, socioeconomico, financeiro, ingresso..."
1,Completo,35,Early Warning + variaveis do 2o semestre (benc...
2,Early Warning sem financeiras,22,Ablacao D3: remove Devedor e MensalidadesEmDia.
3,Early Warning sem macro,21,"Ablacao D7: remove TaxaDesemprego, TaxaInflaca..."
4,Completo sem macro,32,Ablacao D7 no cenario Completo.


## 5. Comparacao de modelos (Fase 2)

Resultados da validacao cruzada estratificada (5 folds), com hiperparametros ajustados
otimizando **F2** (D8).

In [7]:
fase2 = pd.read_csv(config.CV_COMPARISON_CSV)

principal_f2 = (
    fase2[(fase2["target"] == "principal") & (fase2["stage"] == "tuned") & (fase2["metric"] == "f2")]
    [["scenario_label", "model", "n_features", "train_mean", "cv_mean", "cv_std", "gap"]]
    .sort_values("cv_mean", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
display(principal_f2.round(4))

,scenario_label,model,n_features,train_mean,cv_mean,cv_std,gap
0,Completo,logistic_balanced,35,0.8115,0.7945,0.0181,0.0170
1,Completo sem macro,logistic_balanced,32,0.8088,0.7913,0.0191,0.0174
2,Completo sem macro,random_forest_balanced,32,0.8212,0.7867,0.0250,0.0345
3,Completo,random_forest_balanced,35,0.8236,0.7800,0.0256,0.0436
4,Early Warning,random_forest_balanced,24,0.8302,0.7721,0.0175,0.0581
5,Early Warning,logistic_balanced,24,0.7822,0.7712,0.0200,0.0110
6,Early Warning sem macro,logistic_balanced,21,0.7801,0.7697,0.0231,0.0104
7,Early Warning sem macro,random_forest_balanced,21,0.8225,0.7693,0.0252,0.0532
8,Early Warning sem financeiras,logistic_balanced,22,0.7549,0.7487,0.0113,0.0062
9,Completo,hist_gradient_boosting,35,0.8908,0.7423,0.0314,0.1485


In [8]:
dummy = principal_f2[principal_f2["model"] == "dummy_prior"]
if not dummy.empty:
    print(f"Baseline trivial (dummy): F2 = {dummy.iloc[0]['cv_mean']:.4f}")
print("Melhor modelo:", principal_f2.iloc[0]["model"], "| cenario:", principal_f2.iloc[0]["scenario_label"])

rf = principal_f2[principal_f2["model"] == "random_forest_balanced"]
if not rf.empty:
    linha = rf.iloc[0]
    print()
    print("Exemplo de OVERFITTING (Random Forest balanceado):")
    print(f"   F2 no treino = {linha['train_mean']:.4f}")
    print(f"   F2 na validacao = {linha['cv_mean']:.4f}")
    print(f"   gap = {linha['gap']:.4f}")

Melhor modelo: logistic_balanced | cenario: Completo

Exemplo de OVERFITTING (Random Forest balanceado):
   F2 no treino = 0.8212
   F2 na validacao = 0.7867
   gap = 0.0345


## 6. Modelo final e avaliacao no teste (Fase 3)

O modelo foi escolhido pelo criterio da secao 13 do plano e o **limiar de decisao foi
definido na validacao**, nunca no teste (D8). O teste (1.327 alunos) foi avaliado **uma
unica vez**.

In [9]:
metadados = json.loads(config.MODEL_METADATA_PATH.read_text(encoding="utf-8"))

print("Modelo ..........:", metadados["model_name"])
print("Cenario .........:", metadados["scenario_label"])
print("Hiperparametros .:", metadados["best_params"])
print("Limiar ...........:", metadados["decision_threshold"])
print("Justificativa ...:", metadados["selection_justification"]["reason"])

Modelo ..........: logistic_balanced
Cenario .........: Early Warning
Hiperparametros .: {'model__C': '0.1'}
Limiar ...........: 0.4
Justificativa ...: Empate tecnico dentro da margem de 0.01 em F2; escolhido o modelo com menor gap treino-CV (mais estavel e mais interpretavel).


In [10]:
# reconstroi exatamente a mesma divisao congelada e avalia o modelo salvo
modelo = joblib.load(config.MODEL_PATH)
treino_idx, teste_idx, _ = create_or_load_split(df)
_, _, X_teste, y_teste = split_train_holdout(df, "target", treino_idx, teste_idx)
X_teste = X_teste[metadados["input_columns"]]

limiar = metadados["decision_threshold"]
probabilidades = modelo.predict_proba(X_teste)[:, 1]
metricas_teste = compute_threshold_metrics(y_teste, probabilidades, limiar)

pd.Series(metricas_teste).to_frame("valor").round(4)

,valor
threshold,0.4000
accuracy,0.8229
precision,0.6772
recall,0.8568
f1,0.7565
f2,0.8136
roc_auc,0.9071
tn,727.0000
fp,174.0000
fn,61.0000


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
matriz = np.array([[metricas_teste["tn"], metricas_teste["fp"]],
                   [metricas_teste["fn"], metricas_teste["tp"]]])
im = axes[0].imshow(matriz, cmap="Blues")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, matriz[i, j], ha="center", va="center", fontsize=15,
                     color="white" if matriz[i, j] > matriz.max() / 2 else "black")
axes[0].set_xticks([0, 1], ["Previsto nao evasao", "Previsto evasao"])
axes[0].set_yticks([0, 1], ["Real nao evasao", "Real evasao"])
axes[0].set_title(f"Matriz de confusao (limiar {limiar:.2f})")

fpr, tpr, _ = roc_curve(y_teste, probabilidades)
axes[1].plot(fpr, tpr, linewidth=2, label=f"AUC = {metricas_teste['roc_auc']:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="grey")
axes[1].set_title("Curva ROC")
axes[1].set_xlabel("Falsos positivos")
axes[1].set_ylabel("Recall (evasao)")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Evasoes nao detectadas (falsos negativos): {metricas_teste['fn']}")
print(f"Alarmes falsos (falsos positivos)........: {metricas_teste['fp']}")

Evasoes nao detectadas (falsos negativos): 61
Alarmes falsos (falsos positivos)........: 174


/tmp/ipykernel_200314/810913680.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Diagnostico de overfitting / underfitting

Comparamos os **tres niveis**: treino, validacao cruzada e teste. O gap entre treino e
validacao e o principal indicador.

In [12]:
diagnostico = metadados["diagnosis"]
niveis = pd.DataFrame([
    {"nivel": "Treino", **{m: metadados["train_metrics"][m] for m in ("f2", "recall", "precision")},
     "roc_auc": metadados["train_metrics"]["roc_auc"]},
    {"nivel": "Validacao cruzada", "f2": metadados["cv_metrics"]["f2"]["mean"],
     "recall": metadados["cv_metrics"]["recall"]["mean"],
     "precision": metadados["cv_metrics"]["precision"]["mean"],
     "roc_auc": metadados["cv_metrics"]["roc_auc"]["mean"]},
    {"nivel": "Teste (uma vez)", "f2": metricas_teste["f2"], "recall": metricas_teste["recall"],
     "precision": metricas_teste["precision"], "roc_auc": metricas_teste["roc_auc"]},
])
display(niveis.round(4))

print("Diagnostico:", diagnostico["diagnosis"])
print(diagnostico["explanation"])
print()
print(f"gap treino-CV = {diagnostico['gap_train_cv']:.4f}")
print(f"gap CV-teste  = {diagnostico['gap_cv_test']:.4f}")

,nivel,f2,recall,precision,roc_auc
0,Treino,0.8070,0.8481,0.6760,0.9013
1,Validacao cruzada,0.7712,0.7817,0.7336,0.8933
2,Teste (uma vez),0.8136,0.8568,0.6772,0.9071


Diagnostico: ajuste_adequado
Treino e validacao ficam proximos e em nivel util: nao ha sinal relevante de overfitting nem de underfitting.

gap treino-CV = 0.0358
gap CV-teste  = -0.0424


In [13]:
# curvas geradas pelo script da Fase 3
from IPython.display import Image, display as display_image

for arquivo in ("phase3_curva_aprendizado.png", "phase3_curva_validacao_C.png"):
    caminho = config.FIGURES_DIR / arquivo
    if caminho.exists():
        print(arquivo)
        display_image(str(caminho))
    else:
        print(f"(figura ausente: {arquivo})")

phase3_curva_aprendizado.png


'/var/www/html/tech-challenge-fase3/reports/figures/phase3_curva_aprendizado.png'

phase3_curva_validacao_C.png


'/var/www/html/tech-challenge-fase3/reports/figures/phase3_curva_validacao_C.png'

## 8. Demonstracao de inferencia

A mesma funcao usada pelo app Streamlit. Repare que o app **nao decide** nada sobre o
estudante: ele apresenta o **risco** e sugere uma acao (D10).

In [14]:
from src.predict import predict_risk

perfil_risco = {
    "EstadoCivil": "Solteiro", "Curso": "Enfermagem",
    "QualificacaoAnterior": "Ensino Secundario", "Nacionalidade": "Portugues",
    "Genero": "Masculino", "NecessidadesEspeciais": 0, "Devedor": 1,
    "MensalidadesEmDia": 0, "Bolsista": 0, "International": 0,
    "NotaAdmissao": 110.0, "QualificacaoAnteriorGrau": 120.0,
    "UnidadesCurriculares1SemestreCreditado": 0,
    "UnidadesCurriculares1SemestreInscrito": 6,
    "UnidadesCurriculares1SemestreAvaliacoes": 2,
    "UnidadesCurriculares1SemestreAprovado": 0,
    "UnidadesCurriculares1SemestreGrau": 0.0,
    "UnidadesCurriculares1SemestreSemAvaliacoes": 4,
    "TaxaDesemprego": 12.7, "TaxaInflacao": 0.5, "PIB": 1.74,
}
perfil_baixo = {
    **perfil_risco, "Devedor": 0, "MensalidadesEmDia": 1, "Bolsista": 1,
    "NotaAdmissao": 160.0, "UnidadesCurriculares1SemestreAprovado": 6,
    "UnidadesCurriculares1SemestreGrau": 15.5,
    "UnidadesCurriculares1SemestreAvaliacoes": 9,
    "UnidadesCurriculares1SemestreSemAvaliacoes": 0,
}

for nome, perfil in (("Perfil com sinais de risco", perfil_risco),
                     ("Perfil com bom desempenho", perfil_baixo)):
    resultado = predict_risk(perfil)
    print(f"{nome}:")
    print(f"   probabilidade de evasao = {resultado['probability']:.1%}")
    print(f"   faixa de risco          = {resultado['risk_band']}")
    print(f"   recomendacao            = {resultado['recommendation']}")
    print()

Perfil com sinais de risco:
   probabilidade de evasao = 98.7%
   faixa de risco          = Alto
   recomendacao            = Priorizar contato e acao de retencao.

Perfil com bom desempenho:
   probabilidade de evasao = 10.6%
   faixa de risco          = Baixo
   recomendacao            = Manter acompanhamento padrao.



## 9. Conclusoes

1. **A base tinha um artefato serio**: 3.465 notas fora da escala 0-20, corrigidas por
   regra validada (100% dos valores caem em [0, 20]; nenhuma inconsistencia com o numero
   de disciplinas aprovadas).
2. **`class_weight="balanced"` foi a maior alavanca** do projeto: +7 p.p. de F2 na
   Regressao Logistica, mais do que qualquer ajuste de hiperparametro.
3. **A Regressao Logistica balanceada venceu** em todos os cenarios testados, com o menor
   gap treino-validacao — modelo simples, estavel e interpretavel.
4. **O Random Forest padrao e um caso didatico de overfitting** (F2 de treino = 1,0000
   contra 0,7419 na validacao); o ajuste de profundidade corrigiu o problema e ainda
   melhorou a generalizacao.
5. **O cenario Completo supera o Early Warning em apenas ~2,3 p.p.**, ao custo de esperar
   o 2o semestre - por isso o **Early Warning** foi escolhido para o deploy (D2).
6. **As variaveis financeiras agregam sinal real** (-2,25 p.p. ao remove-las), mas a
   dependencia temporal segue documentada como **hipotese**, nunca como fato (D3).
7. **As macroeconomicas quase nao agregam** (-0,15 p.p.) e carregam risco de vazamento
   temporal (D7).
